# Eigendecomposition

Eigenvectors are the directions a linear transformation stretches without
rotating; eigenvalues are how much it stretches them along each one. This
notebook uses the repo's own `LinearAlgebra.eigendecomposition` (see
`src/math_utils/linear_algebra.py`) to compute both, then visualizes what
they mean geometrically and why they matter for AI/ML (PCA, spectral
clustering, PageRank, stability of recurrent networks).

## Table of Contents
1. [Computing eigenvalues/eigenvectors](#compute)
2. [Geometric intuition: transforming a unit circle](#geometry)
3. [Why AI cares: PCA via eigendecomposition of a covariance matrix](#pca)


<a id='compute'></a>
## 1. Computing eigenvalues/eigenvectors

In [ ]:
import sys
sys.path.insert(0, "../../src")

import numpy as np
import matplotlib.pyplot as plt
from math_utils.linear_algebra import LinearAlgebra

A = np.array([[4.0, 1.0],
              [2.0, 3.0]])

eigenvalues, eigenvectors = LinearAlgebra.eigendecomposition(A)
print("Eigenvalues:", eigenvalues)
print("Eigenvectors (columns):\n", eigenvectors)

# Sanity check: A @ v == lambda * v for each eigenpair
for i in range(len(eigenvalues)):
    v = eigenvectors[:, i]
    lhs = A @ v
    rhs = eigenvalues[i] * v
    assert np.allclose(lhs, rhs), f"Eigenpair {i} failed A@v = lambda*v"
print("A @ v == lambda * v holds for every eigenpair.")


<a id='geometry'></a>
## 2. Geometric intuition: transforming a unit circle

Every other direction gets rotated *and* scaled by `A`; only the eigenvector directions get scaled in place. Plotting `A` applied to a circle of unit vectors makes this visible - the ellipse's axes line up exactly with the eigenvectors, and the axis lengths are the eigenvalues.

In [ ]:
theta = np.linspace(0, 2 * np.pi, 200)
circle = np.stack([np.cos(theta), np.sin(theta)])  # unit circle, 2 x N
transformed = A @ circle

fig, ax = plt.subplots(figsize=(6, 6))
ax.plot(circle[0], circle[1], label="unit circle", alpha=0.5)
ax.plot(transformed[0], transformed[1], label="A @ unit circle")

origin = np.zeros(2)
for i in range(len(eigenvalues)):
    v = eigenvectors[:, i] * eigenvalues[i]
    ax.annotate(
        "", xy=(v[0], v[1]), xytext=origin,
        arrowprops=dict(arrowstyle="->", color=f"C{i+2}", lw=2),
    )
    ax.text(v[0] * 1.1, v[1] * 1.1, f"$\\lambda_{i+1}={eigenvalues[i]:.2f}$")

ax.set_aspect("equal")
ax.axhline(0, color="grey", lw=0.5)
ax.axvline(0, color="grey", lw=0.5)
ax.legend()
ax.set_title("A stretches along its eigenvectors, rotates everything else")
plt.show()


<a id='pca'></a>
## 3. Why AI cares: PCA via eigendecomposition of a covariance matrix

Principal Component Analysis finds the directions of greatest variance in data - those directions *are* the eigenvectors of the data's covariance matrix, and the variance explained along each one is its eigenvalue. This is the same `eigendecomposition` call, just applied to `cov(X)` instead of an arbitrary matrix.

In [ ]:
rng = np.random.default_rng(42)
mean = [0, 0]
cov = [[3.0, 1.5], [1.5, 1.0]]
X = rng.multivariate_normal(mean, cov, size=300)

cov_matrix = np.cov(X.T)
pca_eigenvalues, pca_eigenvectors = LinearAlgebra.eigendecomposition(cov_matrix)

# Sort descending so component 1 is the direction of greatest variance
order = np.argsort(pca_eigenvalues)[::-1]
pca_eigenvalues = pca_eigenvalues[order]
pca_eigenvectors = pca_eigenvectors[:, order]

explained_ratio = pca_eigenvalues / pca_eigenvalues.sum()
print("Explained variance ratio per component:", explained_ratio)

fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(X[:, 0], X[:, 1], alpha=0.3, s=15, label="data")
for i in range(2):
    direction = pca_eigenvectors[:, i] * np.sqrt(pca_eigenvalues[i]) * 2
    ax.annotate(
        "", xy=tuple(direction), xytext=(0, 0),
        arrowprops=dict(arrowstyle="->", color=f"C{i+1}", lw=2),
    )
    ax.text(*(direction * 1.1), f"PC{i+1} ({explained_ratio[i]:.0%} var)")

ax.set_aspect("equal")
ax.set_title("Principal components = eigenvectors of the covariance matrix")
plt.show()


## Takeaways

- `LinearAlgebra.eigendecomposition(A)` returns `(eigenvalues, eigenvectors)`
  such that `A @ eigenvectors[:, i] == eigenvalues[i] * eigenvectors[:, i]`.
- Geometrically: eigenvectors are the only directions a linear map doesn't
  rotate, just scales by the eigenvalue.
- PCA is exactly this decomposition applied to a covariance matrix - no new
  math, same function.
- See also: `notebooks/calculus/01_gradient_descent.ipynb` for how the
  *other* half of the ML toolkit (optimization) builds on `Calculus` in
  this repo.
